# JSON Basics — 08: JSON with REST APIs

Reading JSON responses from REST APIs and processing them in Spark.
Common in data ingestion pipelines that pull from external services.

Topics: reading API JSON responses, paginated responses, unnesting API data,
handling API-specific patterns (meta/data wrappers, pagination cursors).


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Typical API Response Structures



In [ ]:

import json as pyjson, pathlib

# Pattern 1: Single object response
single = {"id":1,"name":"Alice","email":"alice@ex.com","created_at":"2024-01-15T10:30:00Z"}

# Pattern 2: Array response
array = [{"id":i,"name":f"User_{i}","score":round(i*1.5,1)} for i in range(1,101)]

# Pattern 3: Wrapped response (most common in REST APIs)
wrapped = {
    "data":    [{"id":i,"value":round(i*9.99,2),"status":"active"} for i in range(1,201)],
    "meta":    {"total":200,"page":1,"per_page":200,"pages":1},
    "status":  "success",
    "request_id": "req_abc123"
}

# Pattern 4: Paginated — multiple response files
for page in range(1,4):
    response = {
        "data":   [{"id":(page-1)*50+i,"amount":round(i*10.5,2)} for i in range(1,51)],
        "meta":   {"page":page,"total_pages":3,"total_records":150},
        "cursor": f"cursor_{page+1}" if page<3 else None
    }
    path = f"{DATA_DIR}/api_page_{page}.json"
    with open(path,"w") as f: pyjson.dump(response,f)

print("Created API response files:")
for p in range(1,4):
    print(f"  api_page_{p}.json: page {p} of 3 (50 records)")


## Step 2 — Reading Wrapped API Responses



In [ ]:

import glob as glib

# Read all paginated API responses
all_files = sorted(glib.glob(f"{DATA_DIR}/api_page_*.json"))

# Each file is a full JSON object with nested data array
dfs = []
for file_path in all_files:
    df_page = spark.read.option("multiLine",True).json(file_path)
    # Unwrap the data array
    df_data = df_page.select(F.explode("data").alias("record")) \
                     .select("record.*")
    dfs.append(df_data)

from functools import reduce
from pyspark.sql import DataFrame
df_all = reduce(DataFrame.union, dfs)
print(f"All pages combined: {df_all.count():,} records")
df_all.show(5)


## Step 3 — Normalizing API Data



In [ ]:

import json as pyjson, pathlib

# Realistic API response: orders with nested customer + items
api_orders = {
    "orders": [
        {"order_id":i,"customer":{"id":i%100,"name":f"Cust_{i%100}","tier":"gold" if i%5==0 else "standard"},
         "line_items":[{"sku":f"SKU{j}","price":round(j*15.5,2),"qty":j} for j in range(1,i%4+2)],
         "totals":{"subtotal":round(i*25.0,2),"tax":round(i*2.5,2),"shipping":5.99},
         "status":"confirmed","created":"2024-01-{:02d}T12:00:00Z".format(i%28+1)}
        for i in range(1,201)
    ],
    "count":200
}
api_path = f"{DATA_DIR}/api_orders.json"
with open(api_path,"w") as f: pyjson.dump(api_orders, f)

# Read and normalize
df_raw = spark.read.option("multiLine",True).json(api_path)
df_orders = df_raw.select(F.explode("orders").alias("o")) \
                  .select(
                      F.col("o.order_id"),
                      F.col("o.customer.id").alias("customer_id"),
                      F.col("o.customer.name").alias("customer_name"),
                      F.col("o.customer.tier").alias("customer_tier"),
                      F.col("o.totals.subtotal").alias("subtotal"),
                      F.col("o.totals.tax").alias("tax"),
                      F.size("o.line_items").alias("item_count"),
                      F.col("o.status"),
                  )
print(f"Normalized {df_orders.count():,} orders:")
df_orders.show(5)


## Step 4 — From API JSON to Analytics Table



In [ ]:

df_orders.write.mode("overwrite") \
         .option("compression","zstd") \
         .partitionBy("status") \
         .parquet(f"{DATA_DIR}/api_analytics")

df_check = spark.read.parquet(f"{DATA_DIR}/api_analytics")
print(f"Analytics table: {df_check.count():,} rows")
df_check.groupBy("customer_tier","status").agg(
    F.count("*").alias("orders"),
    F.sum("subtotal").alias("revenue")
).orderBy(F.desc("revenue")).show()


## Summary



In [ ]:

print("""
REST API JSON patterns:
  Single object   → spark.read.option("multiLine",True).json(path)
  Array of objects→ spark.read.option("multiLine",True).json(path)
                    then .select(F.explode("array_field").alias("r")).select("r.*")
  Wrapped response→ .select(F.explode("data").alias("r")).select("r.*")
  NDJSON (streaming)→ spark.read.json(path)  # one object per line, default

Paginated API pattern:
  1. Fetch each page to separate JSON file
  2. Read all files with multiLine=True
  3. Unwrap data array with explode()
  4. Union all page DataFrames
  5. Write to Parquet
""")
